# Cleaning: Part-1

The goal now is to post-process the job descriptions you collected in `HW-3.1-crawl.ipynb`.

In this section you MUST do it manually using traditional data science cleaning and parsing skills, i.e. without use of ANY LLM tools or wrappers. (You will do the LLM version in `HW-3.3-cleaning-2.ipynb`, and compare the two in your EDA.)

The unit of analysis in this case is "job", i.e. one row per unique job posting.

Your task is to write a cleaning script that parses the crawl results and creates a data frame, saved to `data/processed-jobs-1.csv`. This data frame will be used for later EDA.

## Getting started

Some tips for working with the raw crawl files:

* Load every JSON file in `data/` (e.g. with `glob`), and collect the entries of each file's `jobs_results` list. Skip the provided `example-out-...` file, or keep it, but make that choice deliberately.
* Each job has useful semi-structured fields in addition to the free-text `description`, e.g. `title`, `company_name`, `location`, `via` (the posting platform), `detected_extensions` (posting age, schedule type, salary, benefits flags), `job_highlights` (qualifications, responsibilities, benefits), and `job_id`. Field availability varies from job to job, so code defensively (e.g. `.get()`).
* `posted_at` values are relative (e.g. "4 days ago"). To convert them to dates, combine them with the time of the crawl, which is available in each file's `search_metadata` (or in your timestamped filename).
* The same job often appears under several search queries. `job_id`, or a combination of title, company, and location, is a good starting point for deduplication.
* Keep track of which search query each job came from; it is a useful feature in its own right.

## Features to extract

Try to extract as many features as possible. Here are some possible suggestions, which together provide a comprehensive overview for analyzing job postings for trends, patterns, and insights.

You will almost certainly not be able to extract all of these features, but do your best to make the most detailed data set that you can. You can also ADD more features to this list if they come to mind.

- **Job Title**
- **Company Name**
- **Sector/Industry**
- **Location** (City, State, Country)
- **Job Type** (Full-time, Part-time, Contract, Internship)
- **Salary** (Range or Exact)
- **Experience Level** (Entry-level, Mid-level, Senior-level)
- **Education Requirements** (Degree, Major, Certifications)
- **Skills/Technologies Required** (e.g., Python, SQL, Machine Learning, Cloud)
- **Job Responsibilities/Duties**
- **Required Years of Experience**
- **Benefits** (Health insurance, Retirement plans, Paid time off)
- **Remote Work Options** (Remote, Hybrid, On-site)
- **Application Deadline**
- **Job Posting Date**
- **Job Description Length** (Number of words or characters)
- **Keywords/Frequency of Terms**
- **Certifications Required or Preferred** (e.g., AWS Certified, PMP)
- **Team Size** (If mentioned)
- **Company Size** (Small, Medium, Large)
- **Company Reputation/Ranking** (If available)
- **Job Posting Platform** (Where the job was posted, e.g., LinkedIn, Indeed)
- **Company Values or Culture** (Diversity, Innovation, Sustainability)
- **Visa Sponsorship Availability**
- **Interview Process Information** (If mentioned)
- **Expected Start Date**
- **Job Posting Expiry Date**
- **Gender Diversity Language** (If any)
- **Working Hours/Shift Type** (Day shift, Night shift, Flexible hours)
- **Required Language Skills**
- **Job Location Proximity to Major Cities** (If provided)
- **Travel Requirements** (Percentage or Frequency)
- **Team Collaboration Tools Mentioned** (Slack, Zoom, etc.)
- **Reporting Line** (e.g., Reports to Senior Manager)
- **Job Benefits Related to Learning & Development** (e.g., Courses, Training)
- **Company Stock Options** (If offered)
- **Required Soft Skills** (e.g., Communication, Leadership)
- **Company Perks** (Gym membership, Free meals, Company car)
- **Job Posting ID or Reference Number**

## Cleaning checklist

While cleaning, try to address the following as best you can:

- **Missing Data**: Identify missing or null values in features like salary, company name, or location.
- **Duplicates**: Check for duplicate job listings or job descriptions.
- **Inconsistent Formatting**: Ensure consistency in categorical variables (e.g., job titles, location formats).
- **Data Type Validation**: Ensure each feature has the appropriate data type (e.g., salary as numeric, dates as datetime).
- **Outliers**: Detect outliers in salary, years of experience, or job description length.

In [ ]:
# INSERT CODE HERE
import glob
import json
import os
import pandas as pd
import re

all_jobs = pd.DataFrame()

extract_columns = [
    "job_highlights",
    "extensions",
    "apply_options"
]

#for file_path in sorted(glob.glob("data/*.json")):
file_path = "data/Artificial-Intelligence-Specialist-20261023-140137.json"
for i in range(1,2):
    try:
        with open(file_path, encoding="utf-8") as f:
            saved = json.load(f)
    except (OSError, UnicodeDecodeError, json.JSONDecodeError) as e:
        print(f"Skipping {file_path}: {e}")
        continue

    jobs = saved.get("jobs_results", [])
    df = pd.json_normalize(saved["jobs_results"])

    for col in extract_columns:
        for row_index, value in df[col].items():
                if not isinstance(value, list):
                    continue
    
                if col == "job_highlights":
                    for section in value:
                        column_name = f"highlights_{section['title']}"

                        if column_name not in df.columns:
                            df[column_name] = ""

                        df.at[row_index, column_name] = "\n".join(
                            section.get("items", [])
                        )
                elif col == "apply_options":
                    for section in value:
                        column_name = f"apply_options_{section['title']}"

                        if column_name not in df.columns:
                            df[column_name] = ""

                        df.at[row_index, column_name] = section.get("link", "")

    df = df.drop(columns = ["job_highlights", "apply_options"], errors = "ignore")
    df.to_csv("test2.csv")

    employment_types = {
            "full-time",
            "part-time",
            "contractor",
            "contract",
            "temporary",
            "internship",
        }

    boolean_columns = {
        "work from home": "work_from_home",
        "no degree mentioned": "no_degree_mentioned",
        "health insurance": "health_insurance",
        "dental insurance": "dental_insurance",
        "paid time off": "paid_time_off",
    }

    # Initialize text and Boolean columns.
    for column in [
        "posted_ago",
        "salary_text",
        "employment_type",
        "other_extensions",
    ]:
        df[column] = ""

    for column in boolean_columns.values():
        df[column] = False

    for row_index, values in df["extensions"].items():
        if not isinstance(values, list):
            continue

        other_extensions = []

        for extension in values:
            if not isinstance(extension, str):
                continue

            text = extension.strip()
            lower = text.lower()

            if (
                re.search(
                    r"\b\d+\+?\s+(?:minute|hour|day|week|month)s?\s+ago\b",
                    lower,
                )
                or lower in {"today", "yesterday"}
            ):
                df.at[row_index, "posted_ago"] = text

            elif (
                re.search(r"\d", text)
                and re.search(
                    r"\b(?:an?|per)\s+(?:hour|day|week|month|year)\b",
                    lower,
                )
            ):
                df.at[row_index, "salary_text"] = text

            elif lower in employment_types:
                df.at[row_index, "employment_type"] = text

            elif lower in boolean_columns:
                column_name = boolean_columns[lower]
                df.at[row_index, column_name] = True

            else:
                other_extensions.append(text)

        df.at[row_index, "other_extensions"] = " | ".join(
            other_extensions
        )

    all_jobs = pd.concat([all_jobs, df], ignore_index=True)

    print(f"Loaded {len(jobs)} jobs from {file_path}")


print(f"Total jobs loaded: {len(all_jobs)}")
all_jobs.to_csv("test.csv")

Loaded 9 jobs from data/Artificial-Intelligence-Specialist-20261023-140137.json
Total jobs loaded: 9
0          100K–130K a year
0                 Full-time
1    125,776–187,093 a year
1                 Full-time
1          Health insurance
2                3 days ago
2         75.2K–129K a year
2                 Full-time
3                7 days ago
3           90K–130K a year
3            Work from home
3                 Full-time
3       No degree mentioned
3          Health insurance
4                3 days ago
4                 Full-time
5               15 days ago
5                 Full-time
6     68,894–103,341 a year
6                 Full-time
6          Dental insurance
6             Paid time off
6          Health insurance
7                8 days ago
7            132,915 a year
7                 Full-time
8                 Full-time
Name: extensions, dtype: str


When you are done, save the result to `data/processed-jobs-1.csv`, and commit your work.